# Reproducibility — load a manifest, inspect runConfig + repeat_group, re-run self_eval_guard

This notebook demonstrates how to:
1. Load a raw JSON manifest from `research/captures/`.
2. Inspect its `runConfig` and `repeat_group` field.
3. Understand `manifestVersion` forward-compatibility semantics.
4. Invoke `self_eval_guard` to surface the self-bias warning path.

**Prerequisites**: `reports/resolver.duckdb` must exist (produced by
`resolver aggregate -tags duckdb`).  The manifest loading in cells 2–4
works without the DB.

In [ ]:
import json
from pathlib import Path

# Locate the first manifest found under research/captures/ that has a
# scorecard sibling in the same run directory.  Falls back to a synthetic
# dict when the repo was cloned without the capture corpus.
CAPTURES_ROOT = Path("../../../research/captures")

manifest_path = None
if CAPTURES_ROOT.exists():
    for manifests_dir in sorted(CAPTURES_ROOT.glob("*/*/manifests")):
        run_dir = manifests_dir.parent
        if (run_dir / "scorecard.json").exists():
            candidates = sorted(manifests_dir.glob("*.json"))
            if candidates:
                manifest_path = candidates[0]
                break

if manifest_path is not None:
    print(f"Using manifest: {manifest_path}")
    m = json.loads(manifest_path.read_text())
else:
    # Fresh-clone fallback: synthetic manifest that matches the real schema.
    # This lets you explore the notebook's logic without a live capture corpus.
    print(
        "No manifests found under research/captures/ — using synthetic fallback.\n"
        "Clone the full corpus or run `resolver run` to populate real captures."
    )
    m = {
        "manifestVersion": 2,
        "runId": "synthetic-001",
        "model": "gresh-general",
        "adapter": None,
        "tokenizerMode": "auto",
        "endpoint": "http://localhost:4000",
        "tier": "1",
        "parallel": 4,
        "scenarioHashes": {},
        "startedAt": "2026-01-01T00:00:00Z",
        "runConfig": {
            "virtualModel": "gresh-general",
            "realModel": "SomeOrg/SomeModel",
            "backend": "vllm",
            "backendPort": 3040,
            "defaultTemperature": 0.6,
            "defaultEnableThinking": True,
            "toolParser": "qwen3_xml",
            "mtp": False,
            "contextSize": 131072,
            "quantization": "fp8",
            "repeatGroup": "group-alpha",
            "notes": "synthetic fallback for notebook demo",
        },
    }

In [ ]:
run_config = m.get("runConfig") or {}

print("runConfig:")
print(json.dumps(run_config, indent=2, default=str))

repeat_group = run_config.get("repeatGroup") or run_config.get("repeat_group")
print(f"\nrepeat_group: {repeat_group!r}")

In [ ]:
# manifestVersion semantics:
#   1  — original schema (pre-runConfig embedding)
#   2  — current schema; runConfig embedded in manifest JSON
#   >2 — forward-compat: the Go ingest layer logs a warning and continues
#         (fields unknown to the current binary are silently ignored).
#
# If you see a version higher than 2 in a real capture, upgrade the
# resolver binary so the ingest path can consume it fully.

CURRENT_SCHEMA_VERSION = 2

manifest_version = m.get("manifestVersion", 1)
print(f"manifestVersion in file : {manifest_version}")
print(f"current SchemaVersion   : {CURRENT_SCHEMA_VERSION}")

if manifest_version > CURRENT_SCHEMA_VERSION:
    print(
        f"WARN: manifestVersion={manifest_version} > {CURRENT_SCHEMA_VERSION}. "
        "The Go ingest path emits a forward-compat warning for this file."
    )
else:
    print("Schema version is current — no forward-compat warning will be emitted.")

In [ ]:
# Demonstrate the self_eval_guard warning path.
# If the reporter model is also one of the models under test, a stderr
# warning is emitted — this cell makes that visible in the notebook.
#
# Note: Store() requires reports/resolver.duckdb to exist.  If it does not,
# this cell prints a friendly message and skips the guard call.

from analyze.report import self_eval_guard
from analyze.db import Store

DB_PATH = Path("../../../reports/resolver.duckdb")

if not DB_PATH.exists():
    print(
        f"DuckDB not found at {DB_PATH}.\n"
        "Run `resolver aggregate -tags duckdb` first to populate it,\n"
        "then re-execute this cell to see the self_eval_guard in action."
    )
else:
    with Store(DB_PATH) as s:
        runs = s.run_summaries()

    # Pass one of the actual run models as the reporter to trigger the warning.
    reporter = runs[0].model if runs else "gresh-general"
    print(f"Calling self_eval_guard with reporter_model={reporter!r} ...")
    print("(warning goes to stderr; Jupyter merges it into the cell output below)")
    import sys
    import io
    buf = io.StringIO()
    old_stderr = sys.stderr
    sys.stderr = buf
    try:
        self_eval_guard(runs, reporter)
    finally:
        sys.stderr = old_stderr
    captured = buf.getvalue()
    if captured:
        print("stderr output:")
        print(captured)
    else:
        print("No warning emitted (reporter model is not in the test set).")

## Done

You have seen how to:
- Locate and load a raw manifest from `research/captures/`.
- Inspect `runConfig` and `repeat_group` to understand which runs belong to a
  reproducibility group.
- Understand `manifestVersion` forward-compatibility semantics.
- Exercise `self_eval_guard` to confirm that self-grading is flagged.

Return to **`quickstart.ipynb`** for the DataFrame-level view of the aggregated data,
or see the project [README](../README.md) for the full workflow.